# Step 1 probe: §5.11 LoRA-v1 on BIPIA indirect injection

**Capstone: Prompt-Injection Defense Evaluation — Notebook 10**

Question: does the §5.11 LoRA fine-tune (trained on direct injection: deepset + neuralchemy + SPML) transfer to BIPIA's indirect injection (email body + user query)?

## Setup

1. Adapter (already in Drive): `MyDrive/capstone_lora/adapters/deberta_v3_base_lora_v1/`
2. Upload these two files to `MyDrive/capstone_lora/data/`:
   - `results/bipia_email_qa_prompts.csv` (800 BIPIA rows: row_id, attack_category, is_attack, full_prompt)
   - `results/bipia_email_qa_results.csv` (off-the-shelf ProtectAI baseline predictions for the same 800 rows)
3. Runtime: **L4 GPU + High-RAM** (T4 also works; A100 unnecessary)
4. Run all (~1-2 min total)

## What it does

Loads `microsoft/deberta-v3-base` + the LoRA-v1 adapter (trained in NB08), runs inference on each BIPIA row's full_prompt, compares to the off-the-shelf ProtectAI baseline that was already measured in §5.8. Reports headline ASR/FAR, per-category recall breakdown, and saves per-row results back to Drive.

Interpretation guide is at the end.

In [1]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/capstone_lora')
ADAPTER_DIR = DRIVE_ROOT / 'adapters' / 'deberta_v3_base_lora_v1'
BIPIA_PROMPTS = DRIVE_ROOT / 'data' / 'bipia_email_qa_prompts.csv'
BIPIA_BASELINE = DRIVE_ROOT / 'data' / 'bipia_email_qa_results.csv'
RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for p in [ADAPTER_DIR, BIPIA_PROMPTS, BIPIA_BASELINE]:
    print(f'  {p}: {"OK" if p.exists() else "MISSING — upload to Drive first"}')

Mounted at /content/drive
  /content/drive/MyDrive/capstone_lora/adapters/deberta_v3_base_lora_v1: OK
  /content/drive/MyDrive/capstone_lora/data/bipia_email_qa_prompts.csv: OK
  /content/drive/MyDrive/capstone_lora/data/bipia_email_qa_results.csv: OK


In [2]:
import os, sys, subprocess
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'

# Same torchao fix as NB08/NB09
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '--quiet', 'torchao'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers>=4.53', 'peft', 'sentencepiece'])
print('Packages installed.')

Packages installed.


In [3]:
import time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

Device: cuda (NVIDIA L4)


In [4]:
print('Loading microsoft/deberta-v3-base + LoRA-v1 adapter...')
base = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/deberta-v3-base',
    num_labels=2,
    id2label={0: 'BENIGN', 1: 'INJECTION'},
    label2id={'BENIGN': 0, 'INJECTION': 1},
)
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))
print('Loaded.')

Loading microsoft/deberta-v3-base + LoRA-v1 adapter...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias        

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loaded.


In [5]:
prompts_df = pd.read_csv(BIPIA_PROMPTS)
baseline_df = pd.read_csv(BIPIA_BASELINE)
print(f'BIPIA prompts: {prompts_df.shape}')
print(f'BIPIA baseline: {baseline_df.shape}')
print(f'Attack rate: {prompts_df["is_attack"].mean():.3f}')

BIPIA prompts: (800, 4)
BIPIA baseline: (800, 16)
Attack rate: 0.938


In [6]:
BATCH = 32
texts = prompts_df['full_prompt'].tolist()
labels = prompts_df['is_attack'].values
row_ids = prompts_df['row_id'].tolist()

preds = []
scores = []
t0 = time.time()
with torch.no_grad():
    for i in range(0, len(texts), BATCH):
        batch = texts[i:i+BATCH]
        inputs = tokenizer(batch, truncation=True, max_length=512, padding=True, return_tensors='pt').to(device)
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        preds.extend((probs > 0.5).astype(int).tolist())
        scores.extend(probs.tolist())
elapsed = time.time() - t0
print(f'LoRA-v1 inference on {len(texts)} BIPIA rows: {elapsed:.1f} sec ({len(texts)/elapsed:.1f} rows/sec)')

LoRA-v1 inference on 800 BIPIA rows: 7.3 sec (109.9 rows/sec)


In [7]:
def metrics_block(y_true, y_pred, name):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    n_attack = int((y_true == 1).sum())
    n_clean = int((y_true == 0).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    asr = 1.0 - (tp / max(n_attack, 1))
    far = fp / max(n_clean, 1)
    print(f'\n=== {name} ===')
    print(f'  Precision: {p:.3f}, Recall: {r:.3f}, F1: {f:.3f}, Acc: {acc:.3f}')
    print(f'  ASR: {asr:.3f}  (= 1 - recall on attacks; lower = better defense)')
    print(f'  FAR: {far:.3f}  (false-alarm rate on clean controls)')
    print(f'  TP={tp}, FP={fp}, FN={fn}, TN={tn}')
    return {'name': name, 'n': len(y_true), 'precision': float(p), 'recall': float(r),
            'f1': float(f), 'accuracy': float(acc), 'asr': float(asr), 'far': float(far),
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

# Off-the-shelf baseline (already measured in §5.8)
baseline_idx = baseline_df.set_index('row_id')
base_pred = np.array([int(baseline_idx.loc[rid, 'deberta_full_prompt_flagged']) for rid in row_ids])

baseline_m = metrics_block(labels, base_pred, 'Off-the-shelf ProtectAI DeBERTa (full_prompt)')
lora_m = metrics_block(labels, np.array(preds), 'LoRA-v1 (trained on direct injection, applied to BIPIA)')


=== Off-the-shelf ProtectAI DeBERTa (full_prompt) ===
  Precision: 0.931, Recall: 0.344, F1: 0.502, Acc: 0.361
  ASR: 0.656  (= 1 - recall on attacks; lower = better defense)
  FAR: 0.380  (false-alarm rate on clean controls)
  TP=258, FP=19, FN=492, TN=31

=== LoRA-v1 (trained on direct injection, applied to BIPIA) ===
  Precision: 0.938, Recall: 1.000, F1: 0.968, Acc: 0.938
  ASR: 0.000  (= 1 - recall on attacks; lower = better defense)
  FAR: 1.000  (false-alarm rate on clean controls)
  TP=750, FP=50, FN=0, TN=0


In [8]:
print('=== Per-attack-category recall comparison ===')
print(f'{"category":<32} {"n":>4} {"off_shelf":>11} {"lora_v1":>9} {"delta":>8}')
cat_rows = []
for cat in sorted(prompts_df['attack_category'].unique()):
    mask = (prompts_df['attack_category'] == cat) & (prompts_df['is_attack'] == 1)
    idx = mask.values
    if idx.sum() < 1:
        continue
    base_recall = (base_pred[idx] == 1).mean()
    lora_recall = (np.array(preds)[idx] == 1).mean()
    delta = lora_recall - base_recall
    cat_rows.append({'category': cat, 'n_attacks': int(idx.sum()),
                     'off_shelf_recall': float(base_recall),
                     'lora_v1_recall': float(lora_recall),
                     'delta_recall': float(delta)})
    print(f'  {cat:<30} {int(idx.sum()):>4} {base_recall:>11.3f} {lora_recall:>9.3f} {delta:>+8.3f}')

=== Per-attack-category recall comparison ===
category                            n   off_shelf   lora_v1    delta
  Base Encoding                    50       0.400     1.000   +0.600
  Business Intelligence            50       0.320     1.000   +0.680
  Conversational Agent             50       0.360     1.000   +0.640
  Emoji Substitution               50       0.340     1.000   +0.660
  Entertainment                    50       0.360     1.000   +0.640
  Information Dissemination        50       0.300     1.000   +0.700
  Language Translation             50       0.340     1.000   +0.660
  Marketing & Advertising          50       0.320     1.000   +0.680
  Misinformation & Propaganda      50       0.360     1.000   +0.640
  Research Assistance              50       0.340     1.000   +0.660
  Reverse Text                     50       0.480     1.000   +0.520
  Scams & Fraud                    50       0.340     1.000   +0.660
  Sentiment Analysis               50       0.300     1.0

In [9]:
print('=== HEADLINE (LoRA-v1 vs off-the-shelf on BIPIA n=800) ===')
print(f'  Off-the-shelf ASR: {baseline_m["asr"]:.3f}  FAR: {baseline_m["far"]:.3f}')
print(f'  LoRA-v1        ASR: {lora_m["asr"]:.3f}  FAR: {lora_m["far"]:.3f}')
print(f'  Delta ASR (negative is better defense): {lora_m["asr"] - baseline_m["asr"]:+.3f}')
print(f'  Delta FAR (negative is fewer false alarms): {lora_m["far"] - baseline_m["far"]:+.3f}')

=== HEADLINE (LoRA-v1 vs off-the-shelf on BIPIA n=800) ===
  Off-the-shelf ASR: 0.656  FAR: 0.380
  LoRA-v1        ASR: 0.000  FAR: 1.000
  Delta ASR (negative is better defense): -0.656
  Delta FAR (negative is fewer false alarms): +0.620


In [10]:
import json

out_df = pd.DataFrame({
    'row_id': row_ids,
    'attack_category': prompts_df['attack_category'],
    'is_attack': labels,
    'off_the_shelf_pred': base_pred,
    'lora_v1_pred': preds,
    'lora_v1_score': scores,
})
(RESULTS_DIR / 'lora_v1_on_bipia.csv').write_text(out_df.to_csv(index=False))

summary = {
    'experiment': 'lora_v1_on_bipia_transfer_test',
    'adapter_source': str(ADAPTER_DIR),
    'n_rows': len(labels),
    'inference_sec': elapsed,
    'rows_per_sec': len(texts) / elapsed,
    'baseline': baseline_m,
    'lora_v1': lora_m,
    'per_category': cat_rows,
    'delta_asr': float(lora_m['asr'] - baseline_m['asr']),
    'delta_far': float(lora_m['far'] - baseline_m['far']),
}
(RESULTS_DIR / 'lora_v1_on_bipia_metrics.json').write_text(json.dumps(summary, indent=2))
print(f'Saved to {RESULTS_DIR}')

Saved to /content/drive/MyDrive/capstone_lora/results


## Interpretation guide

| Delta ASR (LoRA-v1 − baseline) | Reading |
|---|---|
| ≤ -0.10 | Strong transfer — LoRA-v1 generalises to indirect injection. §5.11 narrative extends: in-distribution fine-tuning on direct injection helps indirect injection too. Write up directly. |
| -0.10 to -0.03 | Modest transfer — direct-injection training partially helps indirect. Motivates Step 2 (BIPIA-augmented LoRA) for stronger result. |
| -0.03 to +0.03 | No meaningful transfer. Cross-distribution generalisation fails. Step 2 (BIPIA-specific training) becomes the empirical answer. |
| ≥ +0.03 | LoRA-v1 hurts on BIPIA. Indicates direct-injection training over-specialised in a way that loses indirect-injection signal. |

**After running, download these from Drive to repo:**
- `MyDrive/capstone_lora/results/lora_v1_on_bipia.csv` → `results/lora_v1_on_bipia.csv`
- `MyDrive/capstone_lora/results/lora_v1_on_bipia_metrics.json` → `results/lora_v1_on_bipia_metrics.json`